In [ ]:
!pip install -U peft transformers accelerate bitsandbytes
!pip uninstall -y unsloth
!pip install --upgrade --no-cache-dir \
  "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

# **Qwen**

In [ ]:
import pandas as pd
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import numpy as np

# Paths
TRAIN_PATH = '/kaggle/input/semeval-polarization/test_phase_pola/subtask1/train/ben.csv'
TEST_PATH = '/kaggle/input/semeval-polarization/test_phase_pola/subtask1/test/ben.csv' 

# Configuration
NUM_CLASSES = 2  # Binary classification: 0 (non-polarized) or 1 (polarized)
max_seq_length = 2048

# Load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)



# Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# Load data
train_df = pd.read_csv(TRAIN_PATH)
# train_df = train_df.head(20)
print(f"Train samples: {len(train_df)}")
print(f"Train label distribution:\n{train_df['polarization'].value_counts()}")

# Create prompt template for Qwen3 conversational format
# def create_prompt(text, label=None):
#     """Create a conversational prompt for Qwen3"""
#     system_message = "You are an expert at detecting polarized content in Bengali text. Polarized text contains division, stereotyping, vilification, dehumanization, or intolerance towards groups or their views."
    
#     user_message = f"Classify this Bengali text as polarized (1) or non-polarized (0). Respond with only a single digit: 0 or 1.\n\nText: {text}"
    
#     # Qwen3 chat template format
#     prompt = f"<|im_start|>system\n{system_message}<|im_end|>\n<|im_start|>user\n{user_message}<|im_end|>\n<|im_start|>assistant\n"
    
#     if label is not None:
#         prompt += str(label) + "<|im_end|>"
    
#     return prompt

def create_prompt(text, label=None):
    system_message = (
        "তুমি একজন বিশেষজ্ঞ, যার কাজ হলো বাংলা লেখায় পোলারায়িত (polarized) "
        "মতামত শনাক্ত করা। পোলারায়িত লেখা সাধারণত বিভাজন, দলভিত্তিক ঘৃণা, "
        "স্টেরিওটাইপিং, অপমান, অবমাননা, বা কোনো গোষ্ঠী বা মতাদর্শের বিরুদ্ধে "
        "তীব্র অসহিষ্ণুতা প্রকাশ করে।"
    )

    user_message = (
        "নিচের বাংলা লেখাটি পোলারায়িত (1) নাকি অপোলারায়িত (0) তা নির্ধারণ করো। "
        "শুধুমাত্র একটি সংখ্যা দেবে: 0 অথবা 1।\n\n"
        f"লেখা: {text}"
    )

    prompt = (
        "<|im_start|>system\n"
        f"{system_message}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{user_message}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )

    if label is not None:
        prompt += str(label) + "<|im_end|>"

    return prompt


# Prepare training dataset
def prepare_dataset(df):
    prompts = []
    for _, row in df.iterrows():
        text = row['text']
        label = int(row['polarization'])
        prompt = create_prompt(text, label)
        prompts.append(prompt)
    return prompts

train_prompts = prepare_dataset(train_df)
combined_dataset = Dataset.from_dict({"text": train_prompts})

print(f"\nSample training prompt:\n{train_prompts[0]}")
print(f"\nDataset size: {len(combined_dataset)}")

# Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = combined_dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        warmup_steps = 20,
        # max_steps = 400,  
        learning_rate = 2e-4,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "./polarization_qwen14b",
        # save_strategy = "steps",
        # save_steps = 100,
        report_to = "none",
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
    ),
)

# Train
print("\n" + "="*50)
print("Starting training...")
print("="*50 + "\n")

trainer.train()

# Save model
model.save_pretrained("polarization_qwen14b_final")
tokenizer.save_pretrained("polarization_qwen14b_final")

print("\nTraining completed!")

# ==== INFERENCE ON TEST DATA ====
print("\n" + "="*50)
print("Starting inference on test set...")
print("="*50 + "\n")

# Load test data
test_df = pd.read_csv(TEST_PATH)
# test_df = test_df.head(20)
print(f"Test samples: {len(test_df)}")

# Prepare model for inference
FastLanguageModel.for_inference(model)

@torch.no_grad()
def predict_single(text, model, tokenizer):
    prompt = create_prompt(text, label=None)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 1,          # IMPORTANT
        temperature = 0.,
        do_sample = False,
        pad_token_id = tokenizer.eos_token_id,
        eos_token_id = tokenizer.eos_token_id,
    )

    # Decode only newly generated token(s)
    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    # Extract 0 or 1 safely
    m = re.search(r"[01]", gen_text)
    return int(m.group()) if m else 0


# Make predictions on test set
predictions = []
for idx, row in test_df.iterrows():
    text = row['text']
    pred = predict_single(text, model, tokenizer)
    predictions.append(pred)
    
    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(test_df)} samples...")

# Create submission dataframe
submission_df = pd.DataFrame({
    'id': test_df['id'],
    'polarization': predictions
})

# Save submission file
submission_df.to_csv('submission.csv', index=False)

print(f"\nPredictions completed!")
print(f"Submission file saved as 'submission.csv'")
print(f"\nPrediction distribution:")
print(submission_df['polarization'].value_counts())
print(f"\nClass balance:")
print(f"Non-polarized (0): {(submission_df['polarization'] == 0).sum()} ({(submission_df['polarization'] == 0).sum() / len(submission_df) * 100:.2f}%)")
print(f"Polarized (1): {(submission_df['polarization'] == 1).sum()} ({(submission_df['polarization'] == 1).sum() / len(submission_df) * 100:.2f}%)")

# Display first few predictions
print(f"\nFirst 10 predictions:")
print(submission_df.head(10))

# Optional: Save some example predictions with text for manual inspection
sample_results = pd.DataFrame({
    'id': test_df['id'].head(20),
    'text': test_df['text'].head(20),
    'prediction': predictions[:20]
})
sample_results.to_csv('sample_predictions.csv', index=False)
print(f"\nSample predictions saved to 'sample_predictions.csv' for inspection")